# 独立子图生成 — Fig2~Fig8 逐面板单独输出

本 Notebook 将 Fig2~Fig8 的每个面板拆分为独立的 PNG + PDF 文件，
存放在各自的 FigXX 文件夹下。
运行顺序不可变：
1. 初始化（数据加载 + 模型训练）
2. Fig2 → Fig8 逐面板生成

每个面板输出命名：`Fig02_panel_a_label_availability.png/pdf` 形式。

In [ ]:
from __future__ import annotations
import json, math, os, warnings
from pathlib import Path
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import patches
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr, wilcoxon
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, learning_curve, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.feature_selection import mutual_info_regression

warnings.filterwarnings('ignore')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
# ────── 路径 ──────
NOTEBOOK_DIR = Path.cwd()
# 自动识别 notebook 所在目录
if NOTEBOOK_DIR.name == 'Fig00_scripts':
    ROOT = NOTEBOOK_DIR.parents[1]
elif (NOTEBOOK_DIR / '新论文绘图_吸附能主线').exists():
    ROOT = NOTEBOOK_DIR
else:
    ROOT = NOTEBOOK_DIR

DATA_PATH = ROOT / 'Comprehensive Dataset on MXene Properties for Catalyst Design (2024).xlsx'
OUT = ROOT / '新论文绘图_吸附能主线'
print(f'ROOT: {ROOT}')
print(f'DATA: {DATA_PATH}')
print(f'OUT:  {OUT}')

In [ ]:
# ────── 全局配置 ──────
RANDOM_STATE = 42

sns.set_theme(style='whitegrid', context='paper')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'font.family': 'Arial',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'axes.titlesize': 14,
    'axes.labelsize': 13,
    'axes.labelweight': 'bold',
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'xtick.major.width': 1.3,
    'ytick.major.width': 1.3,
    'xtick.major.size': 5.5,
    'ytick.major.size': 5.5,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'xtick.bottom': True,
    'ytick.left': True,
    'xtick.top': False,
    'ytick.right': False,
    'legend.fontsize': 10,
    'legend.frameon': False,
    'axes.linewidth': 1.35,
    'axes.unicode_minus': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'mathtext.default': 'regular',
})

In [ ]:
# ────── 调色板与标签 ──────
PALETTE = {
    'blue': '#3C5488', 'cyan': '#4DBBD5', 'teal': '#00A087',
    'green': '#91D1C2', 'gold': '#F0C674', 'orange': '#F39B7F',
    'red': '#E64B35', 'purple': '#8491B4', 'ink': '#2D3E50',
    'muted': '#8897A8', 'light': '#ECF0F5',
}

FEATURE_LABELS = {
    'E_ads_ev': r'E$_{ads}$',
    'E_activation_ev': r'E$_a$',
    'E_reaction_ev': r'E$_{rxn}$',
    'E_Form': r'E$_{form}$',
    'Bader_M1': 'Bader M1', 'Bader_M2': 'Bader M2', 'Bader_X': 'Bader X', 'Bader_T1': 'Bader T1',
    'Bader_T1_missing': 'Bader T1 miss.',
    'Band_gap_PBE_ev': r'E$_g^{PBE}$',
    'a_ang': r'a', 'N_Layers': r'N$_{layers}$',
    'Layer_dist_MX': r'd$_{M-X}$', 'Layer_dist_MM': r'd$_{M-M}$',
    'Length_MX': r'L$_{M-X}$', 'Length_MT1': r'L$_{M-T}$',
    'M1_Z': r'Z$_{M1}$', 'M1_radius': r'r$_{M1}$', 'M1_en': r'EN$_{M1}$',
    'M2_en_missing': 'M2 EN miss.', 'M2_period_missing': 'M2 period miss.',
    'T1_type': 'T1 type', 'T1_type_Z': r'Z$_{T1}$',
    'T1_type_en': r'EN$_{T1}$', 'T1_type_radius': r'r$_{T1}$',
    'T1_type_period_missing': 'T1 period miss.', 'T1_type_radius_missing': 'T1 radius miss.',
    'T1_type_Z_missing': 'T1 Z miss.', 'Length_MT1_missing': r'L$_{M-T}$ miss.',
    'Mol': 'Adsorbate', 'Mol_radical_e': r'radical $e^{-}$',
    'Mol_mw': 'MW', 'Mol_atoms': r'N$_{atoms}$', 'Mol_valence': r'valence $e^{-}$',
}

MODEL_LABELS = {
    'TOPSIS_ensemble': 'Ensemble', 'ExtraTrees': 'ExtraTrees',
    'RF': 'RF', 'GBR': 'GBR', 'HGBR': 'HGBR', 'SVR': 'SVR', 'MLP': 'MLP',
}

MODEL_COLORS = {
    'RF': PALETTE['blue'], 'ExtraTrees': PALETTE['teal'], 'GBR': PALETTE['red'],
    'HGBR': PALETTE['cyan'], 'SVR': PALETTE['orange'], 'MLP': PALETTE['purple'],
    'TOPSIS_ensemble': PALETTE['gold'],
}

ADSORBATE_COLORS = {
    'CO': PALETTE['blue'], 'CO2': PALETTE['cyan'], 'H': PALETTE['orange'],
    'H2': PALETTE['gold'], 'H2O': PALETTE['teal'], 'O': PALETTE['red'],
    'O2': PALETTE['purple'], 'OH': PALETTE['green'],
}

ADSORBATE_LABELS = {
    'CO': 'CO', 'CO2': r'CO$_2$', 'H': 'H', 'H2': r'H$_2$',
    'H2O': r'H$_2$O', 'O': 'O', 'O2': r'O$_2$', 'OH': 'OH',
}

REACTION_LABELS = {
    "CO2*  --> CO* + O*": r'CO$_2^*$ -> CO$^*$ + O$^*$',
    'H2 ---> 2H': r'H$_2$ -> 2H',
    'H2O --> HO + H': r'H$_2$O -> HO + H',
}

ELEMENTS = {
    'H': (1, 1, 1, 2.20, 31), 'C': (6, 14, 2, 2.55, 76),
    'N': (7, 15, 2, 3.04, 71), 'O': (8, 16, 2, 3.44, 66),
    'F': (9, 17, 2, 3.98, 57), 'S': (16, 16, 3, 2.58, 105),
    'Cl': (17, 17, 3, 3.16, 102), 'Br': (35, 17, 4, 2.96, 120),
    'Sc': (21, 3, 4, 1.36, 170), 'Ti': (22, 4, 4, 1.54, 160),
    'V': (23, 5, 4, 1.63, 153), 'Cr': (24, 6, 4, 1.66, 139),
    'Mn': (25, 7, 4, 1.55, 139), 'Fe': (26, 8, 4, 1.83, 132),
    'Co': (27, 9, 4, 1.88, 126), 'Ni': (28, 10, 4, 1.91, 124),
    'Cu': (29, 11, 4, 1.90, 132), 'Zn': (30, 12, 4, 1.65, 122),
    'Y': (39, 3, 5, 1.22, 190), 'Zr': (40, 4, 5, 1.33, 175),
    'Nb': (41, 5, 5, 1.60, 164), 'Mo': (42, 6, 5, 2.16, 154),
    'Tc': (43, 7, 5, 1.90, 147), 'Ru': (44, 8, 5, 2.20, 146),
    'Rh': (45, 9, 5, 2.28, 142), 'Pd': (46, 10, 5, 2.20, 139),
    'Ag': (47, 11, 5, 1.93, 145), 'Hf': (72, 4, 6, 1.30, 175),
    'Ta': (73, 5, 6, 1.50, 170), 'W': (74, 6, 6, 2.36, 162),
    'Re': (75, 7, 6, 1.90, 151), 'Os': (76, 8, 6, 2.20, 144),
    'Ir': (77, 9, 6, 2.20, 141), 'Pt': (78, 10, 6, 2.28, 136),
    'Au': (79, 11, 6, 2.54, 136),
}
ALIASES = {'Rd': 'Rh'}

MOL_PROPS = {
    'H': (1.008, 1, 1, 1), 'H2': (2.016, 2, 2, 0),
    'H2O': (18.015, 3, 8, 0), 'CO': (28.010, 2, 10, 0),
    'CO2': (44.010, 3, 16, 0), 'OH': (17.007, 2, 7, 1),
    'O': (15.999, 1, 6, 2), 'O2': (31.998, 2, 12, 2),
}

In [ ]:
# ────── 辅助函数 ──────
def label_feature(name):
    return FEATURE_LABELS.get(str(name), str(name).replace('_missing', ' miss.').replace('_', ' '))

def label_model(name):
    return MODEL_LABELS.get(str(name), str(name))

def style_axes(ax, tick_size=12, label_size=13, title_size=14, show_ticks=True):
    if show_ticks:
        ax.tick_params(axis='both', which='major', labelsize=tick_size, width=1.45, length=6.0, direction='out', bottom=True, left=True, top=False, right=False, color=PALETTE['ink'], labelcolor=PALETTE['ink'])
    else:
        ax.tick_params(axis='both', which='major', labelsize=tick_size, width=0, length=0, bottom=False, left=False, top=False, right=False, labelcolor=PALETTE['ink'])
    ax.xaxis.label.set_size(label_size)
    ax.yaxis.label.set_size(label_size)
    ax.xaxis.label.set_weight('bold')
    ax.yaxis.label.set_weight('bold')
    ax.title.set_size(title_size)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight('bold')
    for spine in ax.spines.values():
        spine.set_linewidth(1.35)

def save_subfig(fig, fig_no, sub_name, dpi=600):
    """Save a single sub-panel figure as both PNG and PDF."""
    folder = OUT / f'Fig{fig_no:02d}'
    folder.mkdir(parents=True, exist_ok=True)
    png = folder / f'Fig{fig_no:02d}_{sub_name}.png'
    pdf = folder / f'Fig{fig_no:02d}_{sub_name}.pdf'
    fig.savefig(png, bbox_inches='tight', facecolor='white', dpi=dpi)
    try:
        fig.savefig(pdf, bbox_inches='tight', facecolor='white', dpi=dpi)
    except PermissionError:
        print(f'  ! PDF locked, skipped: {pdf.name}')
    plt.close(fig)
    print(f'  ✓ {png.name}')

def elem_tuple(symbol):
    if pd.isna(symbol): return (np.nan,) * 5
    symbol = str(symbol).strip()
    symbol = ALIASES.get(symbol, symbol)
    if symbol in ELEMENTS: return ELEMENTS[symbol]
    if symbol in {'OH', 'HO'}: return tuple(np.nanmean([ELEMENTS['O'], ELEMENTS['H']], axis=0))
    if symbol == 'NH': return tuple(np.nanmean([ELEMENTS['N'], ELEMENTS['H']], axis=0))
    return (np.nan,) * 5

def mol_tuple(mol):
    if pd.isna(mol): return (np.nan,) * 4
    return MOL_PROPS.get(str(mol).strip(), (np.nan,) * 4)

NUMERIC_BASE = ['a_ang', 'N_Layers', 'Layer_dist_MX', 'Layer_dist_MM',
    'Length_MX', 'Length_MT1', 'E_Form', 'Bader_M1', 'Bader_M2',
    'Bader_X', 'Bader_T1', 'Band_gap_PBE_ev']
CATEGORICAL_BASE = ['Stacking', 'M1', 'M2', 'X', 'T1_type', 'Mol']

In [ ]:
# ────── 数据加载 ──────
def load_data():
    wgs = pd.read_excel(DATA_PATH, sheet_name='WGS').replace('-', np.nan)
    dop = pd.read_excel(DATA_PATH, sheet_name='dopants').replace('-', np.nan)
    for df in (wgs, dop):
        for col in df.columns:
            if col not in ['Name','Formula','Stacking','M1','M2','Dopant','X','T1_type',
                           'T2_type','Mol','SMILES','Reaction','DOI_1(Energies)',
                           'DOI_2(Band_gap_HSE06_ev)']:
                df[col] = pd.to_numeric(df[col], errors='coerce')
    wgs = wgs[wgs['E_ads_ev'].notna()].copy()
    dop = dop[dop['E_ads_ev'].notna()].copy()
    return wgs, dop

wgs, dop = load_data()
print(f'WGS: {len(wgs)} rows, {wgs["E_ads_ev"].notna().sum()} Eads labels')
print(f'Dop: {len(dop)} rows, {dop["E_ads_ev"].notna().sum()} Eads labels')

In [ ]:
# ────── 特征工程 ──────
def make_features(df):
    x = pd.DataFrame(index=df.index)
    for col in NUMERIC_BASE:
        if col in df.columns:
            x[col] = pd.to_numeric(df[col], errors='coerce')
            x[f'{col}_missing'] = x[col].isna().astype(int)
    for col in CATEGORICAL_BASE:
        if col in df.columns:
            x[col] = df[col].fillna('None').astype(str)
    for col in ['M1','M2','X','T1_type','Dopant']:
        if col in df.columns:
            props = np.array([elem_tuple(v) for v in df[col]])
            for i, pn in enumerate(['Z','group','period','en','radius']):
                x[f'{col}_{pn}'] = props[:, i]
                x[f'{col}_{pn}_missing'] = np.isnan(props[:, i]).astype(int)
    mol_props = np.array([mol_tuple(v) for v in df.get('Mol', pd.Series(index=df.index))])
    for i, pn in enumerate(['mw','atoms','valence','radical_e']):
        x[f'Mol_{pn}'] = mol_props[:, i]
        x[f'Mol_{pn}_missing'] = np.isnan(mol_props[:, i]).astype(int)
    x['has_termination'] = (df.get('T1_type', pd.Series(index=df.index)).notna()).astype(int)
    x['is_doped_domain'] = (df.get('Dopant', pd.Series(index=df.index)).notna()).astype(int)
    return x

def feature_columns(x):
    cat = [c for c in CATEGORICAL_BASE if c in x.columns]
    num = [c for c in x.columns if c not in cat]
    return num, cat

def one_hot_encoder():
    try: return OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=None)
    except TypeError: return OneHotEncoder(handle_unknown='ignore', sparse=False)

def build_preprocessor(x):
    num_cols, cat_cols = feature_columns(x)
    numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', one_hot_encoder())])
    return ColumnTransformer([('num', numeric_pipe, num_cols), ('cat', cat_pipe, cat_cols)],
                             remainder='drop', sparse_threshold=0.0, verbose_feature_names_out=False)

In [ ]:
# ────── 模型训练 ──────
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true)-np.asarray(y_pred))**2)))

def safe_mape(y_true, y_pred):
    denom = np.maximum(np.abs(np.asarray(y_true)), 0.10)
    return float(np.mean(np.abs((np.asarray(y_true)-np.asarray(y_pred))/denom)*100))

def metrics(y_true, y_pred):
    return {'R2': float(r2_score(y_true, y_pred)), 'RMSE': rmse(y_true, y_pred),
            'MAE': float(mean_absolute_error(y_true, y_pred)), 'MAPE': safe_mape(y_true, y_pred)}

def entropy_topsis(metric_df):
    benefit = pd.DataFrame(index=metric_df.index)
    benefit['R2'] = metric_df['R2'].clip(lower=0)
    for col in ['RMSE','MAE','MAPE']:
        benefit[col] = 1/(metric_df[col]+1e-9)
    arr = benefit.to_numpy(dtype=float)
    arr = (arr-arr.min(axis=0))/(arr.max(axis=0)-arr.min(axis=0)+1e-12)
    p = arr/(arr.sum(axis=0,keepdims=True)+1e-12)
    entropy = -(p*np.log(p+1e-12)).sum(axis=0)/np.log(len(metric_df))
    weights = (1-entropy)/np.sum(1-entropy)
    weighted = arr*weights
    ideal = weighted.max(axis=0); nadir = weighted.min(axis=0)
    d_pos = np.sqrt(((weighted-ideal)**2).sum(axis=1))
    d_neg = np.sqrt(((weighted-nadir)**2).sum(axis=1))
    closeness = d_neg/(d_pos+d_neg+1e-12)
    out = metric_df.copy()
    out['TOPSIS_score'] = closeness
    out['ensemble_weight'] = closeness/closeness.sum()
    for name, val in zip(['w_R2','w_RMSE','w_MAE','w_MAPE'], weights):
        out[name] = val
    return out.sort_values('TOPSIS_score', ascending=False)

def make_models(preprocessor):
    raw = {
        'RF': RandomForestRegressor(n_estimators=420, max_depth=10, max_features=0.55, min_samples_leaf=7, min_samples_split=12, random_state=RANDOM_STATE, n_jobs=-1),
        'ExtraTrees': ExtraTreesRegressor(n_estimators=520, max_depth=12, max_features=0.55, min_samples_leaf=5, min_samples_split=10, random_state=RANDOM_STATE, n_jobs=-1),
        'GBR': GradientBoostingRegressor(n_estimators=300, learning_rate=0.035, max_depth=3, min_samples_leaf=5, subsample=0.85, max_features='sqrt', random_state=RANDOM_STATE),
        'HGBR': HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_depth=4, min_samples_leaf=15, l2_regularization=0.1, early_stopping=True, validation_fraction=0.15, random_state=RANDOM_STATE),
        'SVR': SVR(C=5, gamma='scale', epsilon=0.1),
        'MLP': MLPRegressor(hidden_layer_sizes=(64,32), alpha=0.01, learning_rate_init=0.002, max_iter=1200, early_stopping=True, validation_fraction=0.15, n_iter_no_change=20, random_state=RANDOM_STATE),
    }
    return {name: Pipeline([('prep', clone(preprocessor)), ('model', model)]) for name, model in raw.items()}

def prepare_training(wgs, dop):
    x_wgs = make_features(wgs)
    x_dop = make_features(dop)
    for col in x_wgs.columns.difference(x_dop.columns): x_dop[col] = np.nan
    for col in x_dop.columns.difference(x_wgs.columns): x_wgs[col] = np.nan
    all_missing = [c for c in x_wgs.columns if x_wgs[c].isna().all()]
    if all_missing:
        x_wgs = x_wgs.drop(columns=all_missing)
        x_dop = x_dop.drop(columns=all_missing, errors='ignore')
    x_dop = x_dop[x_wgs.columns]
    y = wgs['E_ads_ev'].astype(float)
    strat = wgs['Mol'].fillna('None').astype(str)
    x_train, x_test, y_train, y_test, idx_train, idx_test = train_test_split(
        x_wgs, y, wgs.index, test_size=0.2, random_state=RANDOM_STATE, stratify=strat)
    preprocessor = build_preprocessor(x_wgs)
    models = make_models(preprocessor)
    preds_test = {}; preds_train = {}; preds_dop = {}; rows = []; fitted = {}
    for name, pipe in models.items():
        pipe.fit(x_train, y_train)
        fitted[name] = pipe
        preds_test[name] = pipe.predict(x_test)
        preds_train[name] = pipe.predict(x_train)
        preds_dop[name] = pipe.predict(x_dop)
        rows.append({'Model': name, **metrics(y_test, preds_test[name]),
                     **{f'Train_{k}': v for k,v in metrics(y_train, preds_train[name]).items()}})
    metric_df = pd.DataFrame(rows).set_index('Model')
    topsis = entropy_topsis(metric_df[['R2','RMSE','MAE','MAPE']])
    weights = topsis['ensemble_weight'].reindex(metric_df.index).to_numpy()
    ens_test = np.column_stack([preds_test[m] for m in metric_df.index]) @ weights
    ens_train = np.column_stack([preds_train[m] for m in metric_df.index]) @ weights
    ens_dop = np.column_stack([preds_dop[m] for m in metric_df.index]) @ weights
    preds_test['TOPSIS_ensemble'] = ens_test
    preds_train['TOPSIS_ensemble'] = ens_train
    preds_dop['TOPSIS_ensemble'] = ens_dop
    metric_df.loc['TOPSIS_ensemble',['R2','RMSE','MAE','MAPE']] = list(metrics(y_test, ens_test).values())
    metric_df.loc['TOPSIS_ensemble',['Train_R2','Train_RMSE','Train_MAE','Train_MAPE']] = list(metrics(y_train, ens_train).values())
    best_model = topsis.index[0]
    return {
        'x_wgs': x_wgs, 'x_dop': x_dop, 'x_train': x_train, 'x_test': x_test,
        'y_train': y_train, 'y_test': y_test, 'idx_test': idx_test,
        'wgs_test': wgs.loc[idx_test].copy(), 'dop': dop.copy(), 'fitted': fitted,
        'preds_test': preds_test, 'preds_train': preds_train, 'preds_dop': preds_dop,
        'metrics': metric_df, 'topsis': topsis, 'best_model': best_model,
        'weights': pd.Series(weights, index=metric_df.index[:-1]),
    }

ctx = prepare_training(wgs, dop)
print(f'Best model by TOPSIS: {ctx["best_model"]}')
print(f'Holdout R²: {ctx["metrics"].loc["TOPSIS_ensemble","R2"]:.4f}')

---
# 独立子图生成

## 图2 数据集标签可用性与特征关联概览

In [ ]:
# ── Fig2 所需的数据分析 ──
def numeric_frame_for_analysis(wgs):
    x = make_features(wgs)
    num_cols, _ = feature_columns(x)
    num = x[num_cols].copy()
    y = wgs['E_ads_ev'].astype(float)
    keep = []
    for c in num.columns:
        s = pd.to_numeric(num[c], errors='coerce')
        if s.notna().sum() > 25 and s.nunique(dropna=True) > 2:
            keep.append(c)
    num = num[keep].apply(pd.to_numeric, errors='coerce')
    num = num.fillna(num.median(numeric_only=True))
    num['E_ads_ev'] = y.values
    return num

def top_numeric_features(num, n=9):
    y = num['E_ads_ev'].values
    candidates = [c for c in num.columns if c != 'E_ads_ev']
    pearson = num[candidates].corrwith(num['E_ads_ev']).abs()
    spear = num[candidates].corrwith(num['E_ads_ev'], method='spearman').abs()
    mi = mutual_info_regression(num[candidates], y, random_state=RANDOM_STATE)
    score = pearson.rank(pct=True)+spear.rank(pct=True)+pd.Series(mi, index=candidates).rank(pct=True)
    return list(score.sort_values(ascending=False).head(n).index)

def pairwise_mi(df):
    cols = list(df.columns)
    arr = np.zeros((len(cols), len(cols)))
    for i, c1 in enumerate(cols):
        for j, c2 in enumerate(cols):
            if i == j: arr[i,j] = 1
            elif i < j:
                try: arr[i,j] = arr[j,i] = mutual_info_regression(df[[c1]], df[c2], random_state=RANDOM_STATE)[0]
                except: val = 0
    if arr.max() > 0: arr /= arr.max()
    return pd.DataFrame(arr, index=cols, columns=cols)

num = numeric_frame_for_analysis(wgs)
feats = top_numeric_features(num, 8)
cols = feats + ['E_ads_ev']
print('Top features:', feats)

In [ ]:
# Fig2-a: 标签可用性柱状图
fig, ax = plt.subplots(figsize=(4.2, 4.8))
availability = pd.Series({
    r'E$_{ads}$' + '\nWGS': wgs['E_ads_ev'].notna().sum(),
    r'E$_a$' + '\nWGS': wgs['E_activation_ev'].notna().sum(),
    r'E$_{rxn}$' + '\nWGS': wgs['E_reaction_ev'].notna().sum(),
    'Reaction' + '\nlabels': wgs['Reaction'].notna().sum(),
})
colors = [PALETTE['teal'], PALETTE['orange'], PALETTE['orange'], PALETTE['purple']]
ax.bar(availability.index, availability.values, color=colors, edgecolor='white')
ax.set_ylabel('Valid rows')
ax.set_ylim(0, 650)
for i, v in enumerate(availability.values):
    ax.text(i, v+15, str(int(v)), ha='center', fontsize=9, weight='bold')
style_axes(ax)
save_subfig(fig, 2, 'panel_a_label_availability')

In [ ]:
# Fig2-b: E_ads 分布直方图
fig, ax = plt.subplots(figsize=(4.6, 4.8))
sns.histplot(wgs['E_ads_ev'], bins=32, kde=True, color=PALETTE['blue'], ax=ax)
ax.axvline(wgs['E_ads_ev'].median(), color=PALETTE['red'], lw=1.5, ls='--', label='median')
ax.set_xlabel(r'E$_{ads}$ (eV)')
ax.set_ylabel('Count')
ax.legend(frameon=False)
style_axes(ax)
save_subfig(fig, 2, 'panel_b_eads_histogram')

In [ ]:
# Fig2-c: E_ads 按吸附物箱线图
fig, ax = plt.subplots(figsize=(5.2, 4.8))
mol_order = wgs.groupby('Mol')['E_ads_ev'].median().sort_values().index
sns.boxplot(data=wgs, x='Mol', y='E_ads_ev', order=mol_order, ax=ax,
    palette=[ADSORBATE_COLORS.get(str(m), PALETTE['muted']) for m in mol_order],
    linewidth=0.8, fliersize=2)
ax.set_xlabel('')
ax.set_ylabel(r'E$_{ads}$ (eV)')
ax.set_xticklabels([ADSORBATE_LABELS.get(t.get_text(), t.get_text()) for t in ax.get_xticklabels()])
style_axes(ax)
save_subfig(fig, 2, 'panel_c_eads_by_adsorbate')

In [ ]:
# Fig2-d: Pearson 相关热图
fig, ax = plt.subplots(figsize=(5.2, 5.0))
pear = num[cols].corr(method='pearson')
mat = pear.rename(index=label_feature, columns=label_feature)
sns.heatmap(mat, cmap='vlag', center=0, vmin=-1, vmax=1, square=True,
    cbar_kws={'shrink': 0.68, 'pad': 0.045}, ax=ax, linewidths=0.15, linecolor='white')
ax.tick_params(axis='x', rotation=45, labelsize=10)
ax.tick_params(axis='y', rotation=0, labelsize=10)
for tick in ax.get_xticklabels(): tick.set_ha('right')
style_axes(ax, tick_size=10, label_size=12, show_ticks=False)
save_subfig(fig, 2, 'panel_d_pearson_correlation')

In [ ]:
# Fig2-e: Spearman 相关热图
fig, ax = plt.subplots(figsize=(5.2, 5.0))
spear = num[cols].corr(method='spearman')
mat = spear.rename(index=label_feature, columns=label_feature)
sns.heatmap(mat, cmap='vlag', center=0, vmin=-1, vmax=1, square=True,
    cbar_kws={'shrink': 0.68, 'pad': 0.045}, ax=ax, linewidths=0.15, linecolor='white')
ax.tick_params(axis='x', rotation=45, labelsize=10)
ax.tick_params(axis='y', rotation=0, labelsize=10)
for tick in ax.get_xticklabels(): tick.set_ha('right')
style_axes(ax, tick_size=10, label_size=12, show_ticks=False)
save_subfig(fig, 2, 'panel_e_spearman_correlation')

In [ ]:
# Fig2-f: 互信息热图
fig, ax = plt.subplots(figsize=(5.2, 5.0))
mi_mat = pairwise_mi(num[cols])
mat = mi_mat.rename(index=label_feature, columns=label_feature)
sns.heatmap(mat, cmap='mako', center=None, vmin=0, vmax=1, square=True,
    cbar_kws={'shrink': 0.68, 'pad': 0.045}, ax=ax, linewidths=0.15, linecolor='white')
ax.tick_params(axis='x', rotation=45, labelsize=10)
ax.tick_params(axis='y', rotation=0, labelsize=10)
for tick in ax.get_xticklabels(): tick.set_ha('right')
style_axes(ax, tick_size=10, label_size=12, show_ticks=False)
save_subfig(fig, 2, 'panel_f_mutual_information')

## 图3 特征筛选与冗余控制

In [ ]:
# ── Fig3 前置计算 ──
best = ctx['best_model']
pipe = ctx['fitted'][best]
result = permutation_importance(pipe, ctx['x_test'], ctx['y_test'],
    n_repeats=12, random_state=RANDOM_STATE, scoring='r2', n_jobs=-1)
imp = pd.DataFrame({
    'feature': ctx['x_test'].columns,
    'importance': result.importances_mean,
    'std': result.importances_std,
}).sort_values('importance', ascending=False)
print('Top features by permutation importance:', list(imp['feature'].head(14)))

# 共识特征
y_ = num['E_ads_ev']
candidates = [c for c in num.columns if c != 'E_ads_ev']
pear_top = set(num[candidates].corrwith(y_).abs().sort_values(ascending=False).head(10).index)
spear_top = set(num[candidates].corrwith(y_, method='spearman').abs().sort_values(ascending=False).head(10).index)
mi_vals = pd.Series(mutual_info_regression(num[candidates], y_, random_state=RANDOM_STATE), index=candidates)
mi_top = set(mi_vals.sort_values(ascending=False).head(10).index)
consensus = sorted((pear_top & spear_top) | (spear_top & mi_top) | (pear_top & mi_top))
print(f'Consensus features (n={len(consensus)}):', consensus)

# 树形图用数据
candidate = [c for c in imp['feature'].head(18) if c in num.columns]
if len(candidate) < 8:
    candidate = top_numeric_features(num, 14)
corr = num[candidate].corr(method='spearman').fillna(0).clip(-1, 1)
dist = 1 - np.abs(corr)
np.fill_diagonal(dist.values, 0)
Z = linkage(squareform(dist.values, checks=False), method='average')

In [ ]:
# Fig3-a: 置换重要性水平条形图
fig, ax = plt.subplots(figsize=(4.8, 4.6))
top_imp = imp.head(14).iloc[::-1]
ybar = np.arange(len(top_imp))
ax.barh(ybar, top_imp['importance'], xerr=top_imp['std'], color=PALETTE['teal'], alpha=0.9)
ax.set_yticks(ybar)
ax.set_yticklabels([label_feature(f) for f in top_imp['feature']])
ax.set_xlabel(r'Mean R$^2$ decrease')
ax.grid(axis='x', alpha=0.22)
ax.xaxis.set_major_locator(MaxNLocator(5))
style_axes(ax, tick_size=10, label_size=12)
save_subfig(fig, 3, 'panel_a_permutation_importance')

In [ ]:
# Fig3-b: 特征聚类树形图
fig, ax = plt.subplots(figsize=(5.2, 5.2))
dendrogram(Z, labels=[label_feature(c) for c in candidate],
    orientation='right', leaf_font_size=8.8,
    color_threshold=np.median(Z[:, 2]),
    above_threshold_color=PALETTE['muted'], ax=ax)
ax.set_xlabel(r'1 - |Spearman $\rho$|')
style_axes(ax, tick_size=9.5, label_size=12)
save_subfig(fig, 3, 'panel_b_feature_dendrogram')

In [ ]:
# Fig3-c: 共识特征韦恩图
fig, ax = plt.subplots(figsize=(5.0, 4.6))
ax.set_aspect('equal')
listed = consensus[:9]
circles = [
    ((0.34, 0.56), 0.285, PALETTE['blue'], 'Pearson', (0.20, 0.90)),
    ((0.66, 0.56), 0.285, PALETTE['orange'], 'Spearman', (0.82, 0.90)),
    ((0.50, 0.36), 0.275, PALETTE['purple'], 'MI', (0.50, 0.68)),
]
for (x, y0), r, color, label, label_pos in circles:
    ax.add_patch(patches.Circle((x, y0), r, color=color, alpha=0.22, ec=color, lw=1.6))
    ax.text(label_pos[0], label_pos[1], label, color=color, ha='center', va='center', fontsize=9.4, weight='bold')
ax.text(0.5, 0.43,
    '\n'.join(label_feature(v) for v in listed) if listed else 'No pairwise consensus',
    ha='center', va='center', fontsize=10.0, color=PALETTE['ink'], linespacing=1.13)
ax.text(0.5, 0.035, f'n = {len(consensus)}', ha='center', fontsize=10, color=PALETTE['muted'], weight='bold')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_axis_off()
save_subfig(fig, 3, 'panel_c_consensus_diagram')

## 图4 六种机器学习模型的吸附能预测基准

In [ ]:
# ── Fig4 前置计算 ──
models_4 = [m for m in ctx['metrics'].index if m != 'TOPSIS_ensemble']
y_test = ctx['y_test'].to_numpy()
y_train = ctx['y_train'].to_numpy()
all_y = [y_train, y_test]
all_pred = []
for m in models_4:
    all_pred.extend([ctx['preds_train'][m], ctx['preds_test'][m]])
lo = min(np.min(v) for v in all_y + all_pred) - 0.6
hi = max(np.max(v) for v in all_y + all_pred) + 0.45
xs = np.array([lo, hi])

In [ ]:
# Fig4-a~f: 各模型散点+残差（组合为一个横向图）
for i, name in enumerate(models_4):
    fig = plt.figure(figsize=(9.0, 5.2))
    gs = fig.add_gridspec(1, 2, width_ratios=[4.15, 1.2], wspace=0.045)
    ax = fig.add_subplot(gs[0, 0])
    ax_res = fig.add_subplot(gs[0, 1], sharey=ax)
    c = MODEL_COLORS.get(name, PALETTE['blue'])
    pred_train = ctx['preds_train'][name]
    pred_test = ctx['preds_test'][name]

    ax.plot([lo, hi], [lo, hi], '--', color=PALETTE['ink'], lw=1.45, alpha=0.65, zorder=1)
    ax.fill_between(xs, xs-0.50, xs+0.50, color='#8A8F98', alpha=0.13, zorder=0)
    ax.scatter(y_train, pred_train, s=44, facecolors='white', edgecolors=c, linewidths=1.25, alpha=0.78, zorder=3, label='Train')
    ax.scatter(y_test, pred_test, s=68, facecolors=c, edgecolors=PALETTE['ink'], linewidths=0.9, marker='^', alpha=0.95, zorder=4, label='Test')
    m = ctx['metrics'].loc[name]
    txt = (f'{label_model(name)}\n'
           f'R$^2_{{train}}$ = {m.Train_R2:.3f}\n'
           f'R$^2_{{test}}$ = {m.R2:.3f}\n'
           f'RMSE$_{{test}}$ = {m.RMSE:.2f}')
    ax.text(0.045, 0.955, txt, transform=ax.transAxes, fontsize=11.5, fontweight='bold',
        va='top', color=PALETTE['ink'],
        bbox=dict(boxstyle='round,pad=0.35', fc='white', ec=c, lw=1.2, alpha=0.96))
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.xaxis.set_major_locator(MaxNLocator(5))
    ax.yaxis.set_major_locator(MaxNLocator(5))
    ax.set_xlabel(r'DFT E$_{ads}$ (eV)')
    ax.set_ylabel(r'Predicted E$_{ads}$ (eV)')
    ax.grid(alpha=0.25, linestyle='--', linewidth=0.75)
    if i == 0:
        ax.legend(loc='lower right', fontsize=9.5, handletextpad=0.25, borderpad=0.2)
    style_axes(ax)

    res_train = pred_train - y_train
    res_test = pred_test - y_test
    ax_res.scatter(res_train, pred_train, s=22, facecolors='white', edgecolors=c, linewidths=0.9, alpha=0.72, zorder=3)
    ax_res.scatter(res_test, pred_test, s=34, facecolors=c, edgecolors=PALETTE['ink'], linewidths=0.75, marker='^', alpha=0.95, zorder=4)
    ax_res.axvline(0, color=PALETTE['muted'], lw=1.0, ls='--', alpha=0.68)
    all_res = np.concatenate([res_train, res_test])
    rmax = max(np.percentile(np.abs(all_res), 98)*1.15, 0.75)
    ax_res.set_xlim(-rmax, rmax)
    ax_res.xaxis.set_major_locator(MaxNLocator(3))
    ax_res.set_xlabel('Error', fontsize=12, fontweight='bold')
    ax_res.tick_params(axis='y', labelleft=False)
    ax_res.grid(alpha=0.25, linestyle='--', linewidth=0.65)
    style_axes(ax_res, tick_size=9, label_size=11)
    save_subfig(fig, 4, f'panel_{chr(97+i)}_{name.lower()}')
    print(f'  → Panel {chr(97+i)}: {name}')

In [ ]:
# Fig4-g: 模型指标对比条形图
metrics_df = ctx['metrics'].loc[models_4 + ['TOPSIS_ensemble']].copy()
metric_plot = metrics_df[['R2','RMSE','MAE']].rename(columns={'R2': r'R$^2$'})
metric_plot.index = [label_model(i) for i in metric_plot.index]
fig, ax = plt.subplots(figsize=(8.0, 4.5))
x = np.arange(len(metric_plot))
width = 0.24
metric_colors = [PALETTE['teal'], '#E6A06A', '#8A7EB8']
for j, (col, color) in enumerate(zip(metric_plot.columns, metric_colors)):
    ax.bar(x+(j-1)*width, metric_plot[col].values, width=width, color=color, edgecolor='white', linewidth=0.6, label=col)
ax.set_xticks(x)
ax.set_xticklabels(metric_plot.index, rotation=27, ha='right')
ax.set_ylabel('Value')
ax.set_ylim(0, max(0.95, float(metric_plot.max().max())*1.12))
ax.yaxis.set_major_locator(MaxNLocator(5))
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=3, fontsize=10, handlelength=1.4, columnspacing=1.2)
ax.grid(axis='y', alpha=0.26)
style_axes(ax, tick_size=11, label_size=12)
save_subfig(fig, 4, 'panel_g_metrics_comparison')

In [ ]:
# Fig4-h: 学习曲线
from sklearn.base import clone as skclone
fig, ax = plt.subplots(figsize=(5.6, 4.5))
lc_model = skclone(ctx['fitted']['ExtraTrees'])
if hasattr(lc_model, 'set_params'):
    lc_model.set_params(model__n_estimators=180, model__n_jobs=-1)
train_sizes = np.linspace(90, len(ctx['x_train']), 5).astype(int)
train_sizes = np.unique(np.clip(train_sizes, 40, len(ctx['x_train'])))
rng = np.random.default_rng(RANDOM_STATE)
records = []
for size in train_sizes:
    for rep in range(5):
        idx = rng.choice(ctx['x_train'].index.to_numpy(), size=size, replace=False)
        estimator = skclone(lc_model)
        estimator.fit(ctx['x_train'].loc[idx], ctx['y_train'].loc[idx])
        records.append({
            'Training samples': int(size),
            'Training': r2_score(ctx['y_train'].loc[idx], estimator.predict(ctx['x_train'].loc[idx])),
            'Validation': r2_score(ctx['y_test'], estimator.predict(ctx['x_test'])),
        })
lc = pd.DataFrame(records)
lc_summary = lc.groupby('Training samples').agg(
    train_mean=('Training','mean'), train_sd=('Training','std'),
    val_mean=('Validation','mean'), val_sd=('Validation','std'))
xs_lc = lc_summary.index.to_numpy(dtype=float)
for mean_col, sd_col, color, label in [
    ('train_mean','train_sd', MODEL_COLORS['ExtraTrees'], 'Training'),
    ('val_mean','val_sd', PALETTE['red'], 'Validation'),
]:
    mean = lc_summary[mean_col].to_numpy(dtype=float)
    sd = lc_summary[sd_col].fillna(0).to_numpy(dtype=float)
    ax.plot(xs_lc, mean, '-o', color=color, lw=1.8, ms=4.5, label=label)
    ax.fill_between(xs_lc, mean-sd, mean+sd, color=color, alpha=0.13, linewidth=0)
ax.set_xlabel('Training samples')
ax.set_ylabel(r'R$^2$')
ymin = max(0.0, float(np.nanmin(lc_summary[['train_mean','val_mean']].values))-0.18)
ax.set_ylim(ymin, 1.04)
ax.xaxis.set_major_locator(MaxNLocator(5))
ax.yaxis.set_major_locator(MaxNLocator(5))
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.26)
style_axes(ax, tick_size=11, label_size=12)
save_subfig(fig, 4, 'panel_h_learning_curve')

## 图5 TOPSIS 集成权重与统计显著性比较

In [ ]:
# ── Fig5 前置计算 ──
models_5 = [m for m in ctx['metrics'].index if m != 'TOPSIS_ensemble']
y5 = ctx['y_test'].to_numpy()
error_df = []
for name in models_5 + ['TOPSIS_ensemble']:
    err = np.abs(y5 - ctx['preds_test'][name])
    error_df.extend({'Model': name, 'AbsError': e} for e in err)
error_df = pd.DataFrame(error_df)
error_df['Model_label'] = error_df['Model'].map(label_model)

pmat = pd.DataFrame(np.ones((len(models_5)+1, len(models_5)+1)),
    index=models_5+['TOPSIS_ensemble'], columns=models_5+['TOPSIS_ensemble'])
for a in pmat.index:
    for b in pmat.columns:
        if a == b: pmat.loc[a,b] = np.nan
        else:
            try: pmat.loc[a,b] = wilcoxon(np.abs(y5-ctx['preds_test'][a]), np.abs(y5-ctx['preds_test'][b])).pvalue
            except ValueError: pmat.loc[a,b] = 1.0

In [ ]:
# Fig5-a: TOPSIS 权重水平条形图
fig, ax = plt.subplots(figsize=(4.4, 4.2))
weights = ctx['weights'].sort_values(ascending=True)
ax.barh([label_model(i) for i in weights.index], weights.values,
    color=[MODEL_COLORS.get(i, PALETTE['gold']) for i in weights.index], edgecolor='white')
ax.set_xlabel('Weight')
ax.set_xlim(0, max(0.34, float(weights.max())+0.045))
for i, v in enumerate(weights.values):
    ax.text(v+0.005, i, f'{v:.2f}', va='center', fontsize=9)
ax.xaxis.set_major_locator(MaxNLocator(5))
style_axes(ax)
save_subfig(fig, 5, 'panel_a_topsis_weights')

In [ ]:
# Fig5-b: 绝对误差分布箱线图
fig, ax = plt.subplots(figsize=(5.6, 4.6))
order = [label_model(i) for i in ctx['metrics'].sort_values('RMSE').index]
model_palette = {label_model(k): v for k,v in MODEL_COLORS.items()}
sns.boxplot(data=error_df, y='Model_label', x='AbsError', order=order, ax=ax,
    palette=model_palette, fliersize=1.8, linewidth=0.8, orient='h')
sns.stripplot(data=error_df.sample(min(len(error_df),420), random_state=RANDOM_STATE),
    y='Model_label', x='AbsError', order=order, ax=ax, color=PALETTE['ink'], alpha=0.25, size=2, orient='h')
ax.set_xlabel(r'|DFT - predicted| E$_{ads}$ (eV)')
ax.set_ylabel('')
ax.xaxis.set_major_locator(MaxNLocator(5))
style_axes(ax)
save_subfig(fig, 5, 'panel_b_error_distribution')

In [ ]:
# Fig5-c: Wilcoxon p-value 热图
fig, ax = plt.subplots(figsize=(5.6, 5.0))
logp = -np.log10(pmat.astype(float))
logp = logp.rename(index=label_model, columns=label_model)
sns.heatmap(logp, cmap='rocket_r', vmin=0, vmax=np.nanpercentile(logp.values, 95),
    ax=ax, cbar_kws={'label': r'$-\log_{10}(p)$', 'pad': 0.03}, linewidths=0.2, linecolor='white')
ax.tick_params(axis='x', rotation=35, labelsize=11)
ax.tick_params(axis='y', rotation=0, labelsize=11)
for tick in ax.get_xticklabels(): tick.set_ha('right')
style_axes(ax, tick_size=12, show_ticks=False)
save_subfig(fig, 5, 'panel_c_wilcoxon_heatmap')

## 图6 最优模型解释性分析

In [ ]:
# ── Fig6 前置计算 ──
def replace_feature_values(x, feature, value):
    xx = x.copy(); xx[feature] = value; return xx

def feature_reference_value(x_train, feature):
    if x_train[feature].dtype == 'object':
        return x_train[feature].mode(dropna=True).iloc[0]
    return float(pd.to_numeric(x_train[feature], errors='coerce').median())

def pdp_curve(pipe_, x_, feature_, grid_size=40):
    s = pd.to_numeric(x_[feature_], errors='coerce')
    grid = np.linspace(s.quantile(0.05), s.quantile(0.95), grid_size)
    vals = []
    sample = x_.sample(min(len(x_), 260), random_state=RANDOM_STATE)
    for g in grid:
        xx = sample.copy(); xx[feature_] = g
        vals.append(float(np.mean(pipe_.predict(xx))))
    return pd.DataFrame({'x': grid, 'pdp': vals})

def ale_curve(pipe_, x_, feature_, bins=12):
    s = pd.to_numeric(x_[feature_], errors='coerce')
    valid = s.notna()
    x_valid = x_.loc[valid].copy()
    s = s.loc[valid]
    qs = np.unique(np.quantile(s, np.linspace(0,1,bins+1)))
    if len(qs) < 4: return pd.DataFrame({'x':[],'ale':[]})
    effects = []; centers = []
    for l, h in zip(qs[:-1], qs[1:]):
        mask = (s>=l)&(s<=h)
        if mask.sum()==0: effects.append(0.0)
        else:
            low = x_valid.loc[mask].copy(); high = x_valid.loc[mask].copy()
            low[feature_] = l; high[feature_] = h
            effects.append(float(np.mean(pipe_.predict(high)-pipe_.predict(low))))
        centers.append((l+h)/2)
    ale = np.cumsum(effects); ale -= np.mean(ale)
    return pd.DataFrame({'x':centers,'ale':ale})

best6 = ctx['best_model']
pipe6 = ctx['fitted'][best6]
top6 = [f for f in imp.query('importance > 0')['feature'].head(8) if f in ctx['x_test'].columns]
if len(top6) < 6: top6 = list(imp['feature'].head(8))
sample6 = ctx['x_test'].sample(min(120, len(ctx['x_test'])), random_state=RANDOM_STATE)
pred6 = pipe6.predict(sample6)
contrib_records = []
for f in top6:
    ref = feature_reference_value(ctx['x_train'], f)
    xx = replace_feature_values(sample6, f, ref)
    delta = pred6 - pipe6.predict(xx)
    raw = sample6[f]
    if raw.dtype == 'object':
        codes = pd.Categorical(raw).codes
        val = (codes-codes.min())/(codes.max()-codes.min()+1e-9)
    else:
        val = pd.to_numeric(raw, errors='coerce')
        val = (val-np.nanmin(val))/(np.nanmax(val)-np.nanmin(val)+1e-9)
    for d, v in zip(delta, val):
        contrib_records.append({'Feature': f, 'Contribution': d, 'ScaledValue': v})
contrib_df = pd.DataFrame(contrib_records)

def is_continuous_feature(f):
    if f.endswith('_missing'): return False
    if f not in ctx['x_test'].columns: return False
    if not pd.api.types.is_numeric_dtype(ctx['x_test'][f]): return False
    return pd.to_numeric(ctx['x_train'][f], errors='coerce').nunique(dropna=True) > 10

continuous_ranked = [f for f in imp['feature'] if is_continuous_feature(f)]
preferred_focus = ['Bader_M1','Bader_X','E_Form','Length_MX','Layer_dist_MX','Length_MT1','N_Layers']
focus = next((f for f in preferred_focus if is_continuous_feature(f)),
             continuous_ranked[0] if continuous_ranked else 'E_Form')
interaction = 'Mol_mw' if 'Mol_mw' in ctx['x_test'].columns else next((f for f in continuous_ranked if f!=focus), focus)
pdp = pdp_curve(pipe6, pd.concat([ctx['x_train'],ctx['x_test']]), focus)
ale = ale_curve(pipe6, pd.concat([ctx['x_train'],ctx['x_test']]), focus)
print(f'Focus feature: {focus}, Interaction: {interaction}')

In [ ]:
# Fig6-a: 类 SHAP 贡献散点图
fig, ax = plt.subplots(figsize=(6.8, 5.0))
order = contrib_df.groupby('Feature')['Contribution'].apply(lambda s: np.mean(np.abs(s))).sort_values().index
ypos = {f: i for i, f in enumerate(order)}
rng = np.random.default_rng(RANDOM_STATE)
for f in order:
    sub = contrib_df[contrib_df['Feature'] == f]
    yj = ypos[f] + rng.normal(0, 0.055, len(sub))
    ax.scatter(sub['Contribution'], yj, c=sub['ScaledValue'], cmap='viridis', s=18, alpha=0.75, edgecolor='none')
ax.axvline(0, color=PALETTE['muted'], lw=0.8)
ax.set_yticks(range(len(order)))
ax.set_yticklabels([label_feature(f) for f in order])
ax.set_xlabel(r'$\Delta$ prediction of E$_{ads}$ (eV)')
sm = plt.cm.ScalarMappable(norm=Normalize(0,1), cmap='viridis')
cbar = fig.colorbar(sm, ax=ax, orientation='horizontal', fraction=0.035, pad=0.105)
cbar.set_label('Feature value (low to high)', fontsize=11, fontweight='bold')
cbar.ax.tick_params(labelsize=11, width=1.25, length=4.5, direction='out', color=PALETTE['ink'], labelcolor=PALETTE['ink'])
ax.xaxis.set_major_locator(MaxNLocator(5))
style_axes(ax)
save_subfig(fig, 6, 'panel_a_contribution_scatter')

In [ ]:
# Fig6-b: 全局置换重要性排序
fig, ax = plt.subplots(figsize=(4.4, 4.5))
top_bar = imp.head(10).iloc[::-1]
ybar = np.arange(len(top_bar))
ax.barh(ybar, top_bar['importance'], color=PALETTE['teal'])
ax.set_yticks(ybar)
ax.set_yticklabels([label_feature(f) for f in top_bar['feature']])
ax.set_xlabel(r'Mean R$^2$ decrease')
ax.xaxis.set_major_locator(MaxNLocator(5))
style_axes(ax, tick_size=10, label_size=13)
save_subfig(fig, 6, 'panel_b_permutation_importance')

In [ ]:
# Fig6-c: PDP 曲线
fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.plot(pdp['x'], pdp['pdp'], color=PALETTE['blue'], lw=2)
ax.fill_between(pdp['x'], pdp['pdp'], pdp['pdp'].mean(), color=PALETTE['blue'], alpha=0.12)
ax.set_xlabel(label_feature(focus))
ax.set_ylabel(r'Mean E$_{ads}$ prediction (eV)')
ax.xaxis.set_major_locator(MaxNLocator(5))
ax.yaxis.set_major_locator(MaxNLocator(5))
style_axes(ax)
save_subfig(fig, 6, 'panel_c_pdp')

In [ ]:
# Fig6-d: ALE 曲线
fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.plot(ale['x'], ale['ale'], color=PALETTE['red'], lw=2)
ax.axhline(0, color=PALETTE['muted'], lw=0.8, ls='--')
ax.set_xlabel(label_feature(focus))
ax.set_ylabel(r'Centered effect on E$_{ads}$ (eV)')
ax.xaxis.set_major_locator(MaxNLocator(5))
ax.yaxis.set_major_locator(MaxNLocator(5))
style_axes(ax)
save_subfig(fig, 6, 'panel_d_ale')

In [ ]:
# Fig6-e: 交互依赖散点图
fig, ax = plt.subplots(figsize=(5.0, 4.2))
if interaction not in sample6.columns:
    interaction = focus
xx = pd.to_numeric(sample6[focus], errors='coerce')
yy = pred6 - pred6.mean()
cc = pd.to_numeric(sample6[interaction], errors='coerce') if interaction in sample6 else xx
sc = ax.scatter(xx, yy, c=cc, cmap='mako', s=32, alpha=0.78, edgecolor='white', linewidth=0.25)
ax.set_xlabel(label_feature(focus))
ax.set_ylabel(r'Centered E$_{ads}$ prediction (eV)')
cb = fig.colorbar(sc, ax=ax, fraction=0.045, pad=0.02)
cb.ax.tick_params(labelsize=11, width=1.25, length=4.5, direction='out', color=PALETTE['ink'], labelcolor=PALETTE['ink'])
ax.xaxis.set_major_locator(MaxNLocator(5))
ax.yaxis.set_major_locator(MaxNLocator(5))
style_axes(ax)
save_subfig(fig, 6, 'panel_e_interaction_dependency')

## 图7 机理敏感性与小样本反应能垒证据

In [ ]:
# ── Fig7 前置计算 ──
def counterfactual_effect(pipe_, x_, feature_, scale=0.5, boot=200):
    s = pd.to_numeric(x_[feature_], errors='coerce')
    sd = s.std()
    if not np.isfinite(sd) or sd == 0: return (0.0, 0.0, 0.0)
    base = pipe_.predict(x_)
    xx = x_.copy(); xx[feature_] = s + scale * sd
    diff = pipe_.predict(xx) - base
    rng = np.random.default_rng(RANDOM_STATE)
    means = [np.mean(diff[rng.integers(0, len(diff), len(diff))]) for _ in range(boot)]
    return float(np.mean(diff)), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

numeric_top = [f for f in imp['feature']
    if f in ctx['x_test'].columns and pd.api.types.is_numeric_dtype(ctx['x_test'][f])
    and not f.endswith('_missing') and pd.to_numeric(ctx['x_train'][f], errors='coerce').nunique(dropna=True) > 8]
effects = []
for f in numeric_top[:8]:
    mean, lo, hi = counterfactual_effect(pipe, ctx['x_test'].copy(), f)
    effects.append({'Feature': f, 'Effect': mean, 'CI_low': lo, 'CI_high': hi})
eff = pd.DataFrame(effects).sort_values('Effect')
barrier = wgs[wgs['E_activation_ev'].notna() & wgs['E_ads_ev'].notna()].copy()
barrier['Reaction_label'] = barrier['Reaction'].map(REACTION_LABELS).fillna(barrier['Reaction'])
print(f'Barrier samples: {len(barrier)}')

In [ ]:
# Fig7-a: 因果 DAG
fig, ax = plt.subplots(figsize=(6.0, 4.8))
G = nx.DiGraph()
edges = [
    ('Composition','Structure'), ('Composition','Electronic state'),
    ('Surface termination','Electronic state'), ('Structure','E_ads'),
    ('Electronic state','E_ads'), ('Adsorbate','E_ads'),
    ('E_ads','Activation barrier'), ('Reaction path','Activation barrier'),
]
G.add_edges_from(edges)
pos = {
    'Composition': (0.14, 0.74), 'Surface termination': (0.14, 0.34),
    'Structure': (0.42, 0.82), 'Electronic state': (0.42, 0.46),
    'Adsorbate': (0.42, 0.12), 'E_ads': (0.68, 0.49),
    'Reaction path': (0.68, 0.16), 'Activation barrier': (0.92, 0.31),
}
labels = {
    'Composition': 'Compo-' + '\nsition', 'Surface termination': 'Surface' + '\nterm.',
    'Structure': 'Structure', 'Electronic state': 'Electronic' + '\nstate',
    'Adsorbate': 'Adsorbate', 'E_ads': r'E$_{ads}$',
    'Reaction path': 'Reaction' + '\npath', 'Activation barrier': 'Activation' + '\nbarrier',
}
node_colors = [PALETTE['blue'], PALETTE['cyan'], PALETTE['green'], PALETTE['green'],
               PALETTE['orange'], PALETTE['red'], PALETTE['purple'], PALETTE['gold']]
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle='-|>', arrowsize=12,
    edge_color=PALETTE['muted'], width=1.3)
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=1500, node_color=node_colors,
    edgecolors='white', linewidths=1.3)
nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=7.4, font_color=PALETTE['ink'], font_weight='bold')
ax.set_axis_off()
ax.set_xlim(0, 1.05); ax.set_ylim(0, 0.96)
save_subfig(fig, 7, 'panel_a_causal_dag')

In [ ]:
# Fig7-b: 反事实影响误差棒图
fig, ax = plt.subplots(figsize=(5.2, 4.6))
y = np.arange(len(eff))
ax.errorbar(eff['Effect'], y,
    xerr=[eff['Effect']-eff['CI_low'], eff['CI_high']-eff['Effect']],
    fmt='o', color=PALETTE['red'], ecolor=PALETTE['muted'], capsize=3)
ax.axvline(0, color=PALETTE['muted'], lw=0.9, ls='--')
ax.set_yticks(y)
ax.set_yticklabels([label_feature(f) for f in eff['Feature']])
ax.set_xlabel(r'Mean $\Delta$E$_{ads}$ prediction (eV)')
ax.xaxis.set_major_locator(MaxNLocator(5))
style_axes(ax)
save_subfig(fig, 7, 'panel_b_counterfactual_effects')

In [ ]:
# Fig7-c: E_ads vs E_a 散点图
fig, ax = plt.subplots(figsize=(5.4, 4.6))
if len(barrier) > 2:
    rho, p = spearmanr(barrier['E_ads_ev'], barrier['E_activation_ev'], nan_policy='omit')
else:
    rho, p = np.nan, np.nan
sns.scatterplot(data=barrier, x='E_ads_ev', y='E_activation_ev',
    hue='Reaction_label', palette=[PALETTE['blue'], PALETTE['orange'], PALETTE['purple']],
    s=42, edgecolor='white', linewidth=0.35, ax=ax)
sns.regplot(data=barrier, x='E_ads_ev', y='E_activation_ev', scatter=False, color=PALETTE['ink'], lowess=True, ax=ax)
ax.set_xlabel(r'E$_{ads}$ (eV)')
ax.set_ylabel(r'E$_a$ (eV)')
p_text = 'p<0.001' if p < 0.001 else f'p={p:.3f}'
ax.text(0.04, 0.96, f'n={len(barrier)}\nSpearman $\\rho$={rho:.2f}\n{p_text}',
    transform=ax.transAxes, va='top', fontsize=11, bbox=dict(fc='white', ec='none', alpha=0.82))
ax.legend(frameon=False, loc='center left', bbox_to_anchor=(1.02, 0.50), fontsize=9, title='Path', title_fontsize=10)
ax.xaxis.set_major_locator(MaxNLocator(5))
ax.yaxis.set_major_locator(MaxNLocator(5))
style_axes(ax)
save_subfig(fig, 7, 'panel_c_ads_vs_barrier')

## 图8 掺杂体系外部迁移诊断与 DFT 标注趋势

In [ ]:
# ── Fig8 前置 ──
PERIODIC_POS = {
    'H': (1,1), 'Sc': (4,3), 'Ti': (4,4), 'V': (4,5), 'Cr': (4,6),
    'Mn': (4,7), 'Fe': (4,8), 'Co': (4,9), 'Ni': (4,10), 'Cu': (4,11),
    'Zn': (4,12), 'Y': (5,3), 'Zr': (5,4), 'Nb': (5,5), 'Mo': (5,6),
    'Tc': (5,7), 'Ru': (5,8), 'Rh': (5,9), 'Pd': (5,10), 'Ag': (5,11),
    'Hf': (6,4), 'Ta': (6,5), 'W': (6,6), 'Re': (6,7), 'Os': (6,8),
    'Ir': (6,9), 'Pt': (6,10), 'Au': (6,11),
}
dop8 = ctx['dop'].copy()
dop8['pred_E_ads'] = ctx['preds_dop']['TOPSIS_ensemble']
dop8['residual'] = dop8['pred_E_ads'] - dop8['E_ads_ev']
dop8['Dopant_clean'] = dop8['Dopant'].replace(ALIASES).fillna('None')
dop8['pred_screening_score'] = np.exp(-((dop8['pred_E_ads']+0.80)/0.90)**2)
dop8['dft_window_score'] = np.exp(-((dop8['E_ads_ev']+0.80)/0.90)**2)
dop_metrics = metrics(dop8['E_ads_ev'], dop8['pred_E_ads'])
rho8, p_rho8 = spearmanr(dop8['E_ads_ev'], dop8['pred_E_ads'], nan_policy='omit')
print(f'Dopant R²={dop_metrics["R2"]:.2f}, RMSE={dop_metrics["RMSE"]:.2f}, Spearman rho={rho8:.2f}')

rank = (dop8.groupby('Dopant_clean').agg(
    n=('E_ads_ev','size'), mean_dft=('E_ads_ev','mean'),
    mean_pred=('pred_E_ads','mean'), transfer_mae=('residual', lambda s: np.mean(np.abs(s))),
    dft_score=('dft_window_score','mean'), pred_score=('pred_screening_score','mean'))
    .query("Dopant_clean != 'None'")
    .sort_values('dft_score', ascending=False))

In [ ]:
# Fig8-a: 迁移散点图（DFT vs 预测）
fig, ax = plt.subplots(figsize=(5.2, 5.0))
lo = min(dop8['E_ads_ev'].min(), dop8['pred_E_ads'].min()) - 0.25
hi = max(dop8['E_ads_ev'].max(), dop8['pred_E_ads'].max()) + 0.25
sns.scatterplot(data=dop8, x='E_ads_ev', y='pred_E_ads', hue='Mol',
    palette=ADSORBATE_COLORS, s=42, edgecolor='white', linewidth=0.3, ax=ax)
ax.plot([lo, hi], [lo, hi], ls='--', color=PALETTE['red'], lw=1.2)
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.xaxis.set_major_locator(MaxNLocator(5))
ax.yaxis.set_major_locator(MaxNLocator(5))
ax.set_xlabel(r'DFT E$_{ads}$ (eV)')
ax.set_ylabel(r'Predicted E$_{ads}$ (eV)')
ax.text(0.04, 0.96,
    f'Direct transfer failed\nR$^2$ = {dop_metrics["R2"]:.2f}\nRMSE = {dop_metrics["RMSE"]:.2f}\nSpearman $\\rho$ = {rho8:.2f}',
    transform=ax.transAxes, va='top', fontsize=10.2, bbox=dict(fc='white', ec='none', alpha=0.84))
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [ADSORBATE_LABELS.get(t,t) for t in labels],
    frameon=False, fontsize=9, loc='lower left', title='Adsorbate',
    title_fontsize=9, handletextpad=0.35, borderpad=0.2)
style_axes(ax)
save_subfig(fig, 8, 'panel_a_transfer_scatter')

In [ ]:
# Fig8-b: 残差按掺杂元素箱线图
fig, ax = plt.subplots(figsize=(6.0, 4.6))
counts = dop8['Dopant_clean'].value_counts()
dop_order = counts[counts >= 3].index
box_df = dop8[dop8['Dopant_clean'].isin(dop_order)].copy()
order = box_df.groupby('Dopant_clean')['residual'].median().sort_values().index
sns.boxplot(data=box_df, x='Dopant_clean', y='residual', order=order,
    palette=sns.color_palette('crest', n_colors=len(order)),
    fliersize=1.5, linewidth=0.8, ax=ax)
ax.axhline(0, color=PALETTE['muted'], lw=0.8, ls='--')
ax.set_xlabel('Dopant', fontsize=9.6, fontweight='bold')
ax.set_ylabel(r'Prediction residual of E$_{ads}$ (eV)', fontsize=8.8, fontweight='bold', labelpad=4)
ax.tick_params(axis='x', rotation=35)
ax.yaxis.set_major_locator(MaxNLocator(5))
style_axes(ax, tick_size=9.5, label_size=8.8)
fig.subplots_adjust(left=0.28, bottom=0.22)
save_subfig(fig, 8, 'panel_b_residual_boxplot')

In [ ]:
# Fig8-c: DFT-window score 水平条形图
fig, ax = plt.subplots(figsize=(4.6, 4.6))
top_rank = rank[rank['n'] >= 3].head(12).iloc[::-1]
ax.barh(top_rank.index, top_rank['dft_score'], color=PALETTE['teal'])
ax.set_xlabel('DFT-window score')
ax.set_xlim(0, 1.08)
ax.xaxis.set_major_locator(MaxNLocator(5))
for i, (idx, row) in enumerate(top_rank.iterrows()):
    ax.text(row['dft_score']+0.012, i, f'n={int(row["n"])}', va='center', fontsize=8, clip_on=False)
style_axes(ax)
save_subfig(fig, 8, 'panel_c_dft_score_barh')

In [ ]:
# Fig8-d: 周期表热图
fig, ax = plt.subplots(figsize=(7.0, 4.2))
cmap = plt.get_cmap('YlGnBu')
norm = Normalize(vmin=rank['dft_score'].min() if len(rank) else 0,
                 vmax=rank['dft_score'].max() if len(rank) else 1)
for element, row in rank.iterrows():
    if element not in PERIODIC_POS: continue
    period, group = PERIODIC_POS[element]
    x = group; y = 7 - period
    c = cmap(norm(row['dft_score']))
    rect = patches.Rectangle((x-0.45, y-0.40), 0.82, 0.72, facecolor=c, edgecolor='white', linewidth=1.0)
    ax.add_patch(rect)
    text_color = 'white' if norm(row['dft_score']) > 0.72 else PALETTE['ink']
    ax.text(x-0.04, y+0.08, element, ha='center', va='center', fontsize=11, weight='bold', color=text_color)
    ax.text(x-0.04, y-0.16, f"{row['dft_score']:.2f}", ha='center', va='center', fontsize=8, color=text_color)
ax.set_xlim(2.3, 12.85); ax.set_ylim(0.35, 3.72)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.015)
cbar.set_label('DFT-window score', fontweight='bold')
cbar.ax.tick_params(labelsize=11, width=1.25, length=4.5, direction='out', color=PALETTE['ink'], labelcolor=PALETTE['ink'])
cbar.ax.yaxis.label.set_size(12)
ax.set_axis_off()
save_subfig(fig, 8, 'panel_d_periodic_heatmap')

---
**全部完成！** 所有独立子图已分别保存至 Fig02~Fig08 对应文件夹。